<h1 id="Python-pour-biochimistes:-Comment-lire,-cr&eacute;er-et-g&eacute;rer-des-fichiers-(ou-des-flux-de-donn&eacute;es)">Python pour biochimistes: Comment lire, cr&eacute;er et g&eacute;rer des fichiers (ou des flux de donn&eacute;es)</h1>
<p>La plus grande partie du travail que nous faisons n&eacute;cessite l'emploi de fichiers: on doit les lire, on doit en &eacute;crire, on doit en modifier. Comment faire en Python? Ce n'Est pas tr&egrave;s compliqu&eacute; mais avant tout, il faut connaitre quelques rudiments de gestion d'un syst&egrave;me de fichiers.</p>
<h2 id="Comment-utiliser-l'Internet-pour-r&eacute;cup&eacute;rer-des-fichiers-simplement?&nbsp;">Comment utiliser l'Internet pour r&eacute;cup&eacute;rer des fichiers simplement?&nbsp;</h2>
<p>Dans l'exemple pr&eacute;c&eacute;dent, on utilise le syst&egrave;me de fichiers local, donc pas besoin d'utiliser une connexion externe. Mais tr&egrave;s souvent, les infos recherch&eacute;es sont ailleurs... Ici encore, la notion de permission est importante car &eacute;videmment on doit avoir, au minimum, les droits de lecture sur le fichier recherch&eacute;. On peut s'y prendre de deux mani&egrave;res diff&eacute;rentes:</p>
<ul>
<li>Si le fichier est par exemple un simple fichier texte sur un serveur Web, on peut utiliser le module <em>requests</em>;</li>
<li>Si le fichier est sur un service de partage de fichier type SFTP, on peut utiliser le module <em>paramiko</em> (<a href="https://docs.paramiko.org/en/2.2/index.html" target="_blank" rel="noopener">https://docs.paramiko.org/en/2.2/index.html </a>) via le protocole SSH v2.&nbsp;</li>
</ul>
<h3>M&eacute;thode requests</h3>
<p>Allons chercher un fichier: la liste des enzymes de restriction avec leur site de coupure. Mais &ccedil;a ne marchera pas... La compagnie New England BioLab semble avoir rendu le t&eacute;l&eacute;chargement direct des donn&eacute;es impossible :-(</p>

In [4]:
import requests

#
# Ça pourrait avoir l'air de ça
# Mais ça va cassé...
#
unURL = "https://vers.un.site.org/unFichier.txt"

response = requests.get(aURL)

aFile = open("../z.misc_files/unFichier.txt","w")

aFile.write(response.content)

aFile.close()


TypeError: write() argument must be str, not bytes

<h3>M&eacute;thode <em>paramiko</em></h3>
<p>Cette m&eacute;thode d&eacute;pend de l'installation du package du m&ecirc;me nom:</p>

In [2]:
!~/Library/Python/3.9/bin/pip install paramiko

Defaulting to user installation because normal site-packages is not writeable


<p>Imaginons un serveur (totalement virtuel pour l'exemple) &agrave; l'adresse 192.168.1.112, accessible &agrave; l'usager bioubuntu, avec le mot de passe identique (pas s&eacute;curitaire du tout mais c'est pour l'exemple!!). Dasn ce compte usager, nous avons le fichier recherch&eacute;, appell&eacute; link_strider.txt. Notre but: le t&eacute;l&eacute;charger et le sauvegarder localement dans z.misc_files</p>

In [10]:
import paramiko

# Variables nécessaires pour la connexion
host = "192.168.1.112"
user = "bioubuntu"
passwd = "bioubuntu"

# Le client SSH est l'outil qui nous permettra d'interagir avec le serveur
client_ssh= paramiko.SSHClient()
client_ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
client_ssh.connect(hostname=host, username=user, password=passwd)

# Effectuons le telechargement
client_sftp = client_ssh.open_sftp()
localFile  = "../z.misc_files/link_strider_remote.txt"
remoteFile = "link_strider.txt"
try:
  client_sftp.get(remoteFile, localFile)
except FileNotFoundError as err:
    print(f"File: {remoteFile} was not found on the source server {self.__host}")
client_sftp.close()